# Copy Config Version with feature changes

Copies `config/{model}/{src_version}` to `{dst_version}` for every model:
- columns in `skip` set to skip (source=manual) and removed from the load list
- `imputation` section added to config.yaml (see `lib/preprocessing.impute_from_defaults`)
- 05b off; runner created in `runners/{dst_version}/`

In [ ]:
src_version = "v3"
dst_version = "v4"
models = None        # None = all config/*/{src_version}; or e.g. ["car_coll"]
overwrite = False    # True = rebuild existing dst folders

skip = [
    "amc_ame_rounded_imps", "amc_mam2_rounded_imps",
    "owt_number_of_titling_transactions_raw",
    "owh_loo_days_raw", "owh_loo_days_imps", "owh_loo_years_imp",
    "cef_est_curr_mi_grp_imps", "cef_ann_mi_grp_raw", "cef_ann_mi_grp_imps",
    "cef_owner_count_raw",
]

In [ ]:
import sys
from pathlib import Path

project_root = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / 'current.pc').exists())
sys.path.insert(0, str(project_root / 'lib'))
from utils import setup_notebook_environment
from version_builder import copy_version, check_copied_version, create_version_runner, IMPUTATION_V4

project_root = setup_notebook_environment()
if models is None:
    models = sorted(p.parent.parent.name for p in Path('config').glob(f'*_*/{src_version}/config.yaml'))
print(f"{len(models)} models: {models}")
print(f"imputation: {IMPUTATION_V4}")

In [ ]:
for m in models:
    copy_version(project_root, m, src_version, dst_version, skip=skip, imputation=IMPUTATION_V4, overwrite=overwrite)
    create_version_runner(project_root, m, src_version, dst_version)
    problems = check_copied_version(project_root, m, src_version, dst_version, skip=skip)
    print(f"   check: {problems or 'OK'}")